# ViSceT5 — Pretrain **gen_all** (decoder read-scene-text, đòn bẩy #1)
Chạy tuần tự. `gen_all` = huấn luyện decoder **sinh scene-text** (khớp đúng đường finetune: encoder chỉ nhận câu hỏi + ảnh + OCR-feature) + MLM/ITM/TWC làm phụ trợ (×0.5) — phần pretrain trực tiếp có ích cho bộ sinh câu trả lời seq2seq.

Sau khi pretrain xong & upload lên HF, dùng `notebooks/finetune_colab.ipynb` để finetune từ nó.

In [ ]:
!git clone https://github.com/Kussssssss/ViSceT5.git
%cd ViSceT5
# QUAN TRỌNG: các thay đổi pretrain (gen_all, vision unfreeze, whole-word mask) nằm ở
# NHÁNH exp/pretrain-gen-all — KHÔNG phải main. Không checkout đúng nhánh sẽ bị lỗi
# 'unrecognized arguments: --vision_unfreeze_last_n --mlm_mask_mode'.
!git fetch origin
!git checkout exp/pretrain-gen-all
!git pull origin exp/pretrain-gen-all
!git log --oneline -1

In [ ]:
%%capture
!bash setup.sh

In [ ]:
import os
HF_PRETRAIN_REPO = 'Kus669/ViSceT5-pretrain-genall-v4'   # run V4 (ITC=image<->OCR + same-image mask)

In [ ]:
import argparse
from scripts import prepare_dataset
prepare_dataset.main(argparse.Namespace(config='configs/data/ViTextVQA.yaml', data_dir='./datasets'))

In [ ]:
from scripts import init_model
init_model.main()

### 1) SMOKE / MOCK — TỰ ĐỘNG in debug đầy đủ (không cần set env)
> ⚠️ Vision unfreeze TẮT, có guard NaN pretrain-only. ✅ MLM chỉ dùng câu hỏi. 📖 Gen = **read-scene-text** (denoise đã bỏ).

Mock **luôn** in debug. Trong log tìm:
1. `>>> [pretrain] ... gen = read-scene-text`
2. `🔬 [VERIFY]` toàn ✅, `✅ [GEN] gen_loss finite & > 0`, KHÔNG có `🚨 ... has NaN`
3. Per-step `[Pretrain] ... Loss(M) Loss(I) Loss(TWC) Loss(GEN)` **giảm dần**
4. `🔎 [MLM DEBUG]`: mỗi từ mask hiện **token thô gold/pred + gộp thành word**
5. `🔧 [GEN DEBUG]`: target (OCR reading) vs output

In [ ]:
import os, importlib
# GÓI V4 (mock): ITC đổi text-side sang CHUỖI OCR (image<->ocr) + same-image mask (image_id)
# + queue 1024; giữ qa-cloze / TWC-hard / unfreeze-2 / diff-LR / best=acc_mlm_grounded.
os.environ['ITC_WEIGHT']='1'
os.environ['ITC_TEXT_SOURCE']='ocr'  # v4: image<->chuỗi-OCR (câu hỏi template nghèo tin -> ITC v3 ghim ln4;
                                     #     OCR định danh ảnh duy nhất + ép CLIP học ĐỌC chữ)
os.environ['ITC_QUEUE']='1024'        # v4: hạ 4096->1024 — đủ khó, thấy tiến độ học sớm hơn
os.environ['ITC_DUP_TAU']='1'         # mask false-neg: trùng NỘI-DUNG-text (hash) HOẶC CÙNG-ẢNH (image_id)
os.environ['TWC_ADV_PROB']='0.6'
os.environ['TWC_DUP_BOX']='0'
os.environ['ITM_WEIGHT']='0'
os.environ['ITM_POLLUTE']='0'
os.environ['GEN_TARGET_STYLE']='qa'
os.environ['VISION_LR_SCALE']='0.1'   # đổi cấu trúc param-group optimizer -> resume PHẢI khớp
os.environ['MLM_RAND_PROB']='0.25'
os.environ.pop('ITC_TEXT_POOL', None) # không dùng ở mode ocr
os.environ.pop('TWC_TRAIN_LOG', None)
from training import pretrain
importlib.reload(pretrain)
# MOCK kiểm: banner itc_text_source=ocr; [diag] ITC loss GIẢM qua step (không ghim ln4);
# không NaN; grounded/random có số; TWC như cũ.
pretrain.main(args_list=[
    'configs/pretrain.yaml',
    '--loss_ablation_mode', 'gen_all',
    '--vision_unfreeze_last_n', '2',
    '--mlm_mask_mode', 'wholeword',
    '--metric_for_best_model', 'acc_mlm_grounded',
    '--smoke_test', 'True',
])

### 2) FULL PRETRAIN — mặc định KHÔNG debug (chỉ progress bar + eval)
Full run **mặc định tắt debug** (chỉ thanh tiến trình train + kết quả eval trên val → tránh đầy output/lag). Muốn **bật debug** cho full: đặt `os.environ['TWC_TRAIN_LOG']='1'` trước khi gọi. Theo dõi `loss_mlm` & `loss_gen` **giảm dần** qua các lần eval.

In [ ]:
import os, importlib
# GÓI V4 (full): ITC image<->OCR (queue 1024, mask trùng-text/cùng-ảnh) + qa-cloze
# + TWC-hard + unfreeze-2 + differential-LR + best-checkpoint = acc_mlm_grounded.
# constant-LR, num_train_epochs=5 CỐ ĐỊNH -> true-resume (Cell 2b) tiếp nối được.
os.environ['ITC_WEIGHT']='1'
os.environ['ITC_TEXT_SOURCE']='ocr'  # v4: image<->chuỗi-OCR (câu hỏi template nghèo tin -> ITC v3 ghim ln4;
                                     #     OCR định danh ảnh duy nhất + ép CLIP học ĐỌC chữ)
os.environ['ITC_QUEUE']='1024'        # v4: hạ 4096->1024 — đủ khó, thấy tiến độ học sớm hơn
os.environ['ITC_DUP_TAU']='1'         # mask false-neg: trùng NỘI-DUNG-text (hash) HOẶC CÙNG-ẢNH (image_id)
os.environ['TWC_ADV_PROB']='0.6'
os.environ['TWC_DUP_BOX']='0'
os.environ['ITM_WEIGHT']='0'
os.environ['ITM_POLLUTE']='0'
os.environ['GEN_TARGET_STYLE']='qa'
os.environ['VISION_LR_SCALE']='0.1'   # đổi cấu trúc param-group optimizer -> resume PHẢI khớp
os.environ['MLM_RAND_PROB']='0.25'
os.environ.pop('ITC_TEXT_POOL', None) # không dùng ở mode ocr
os.environ.pop('TWC_TRAIN_LOG', None)
from training import pretrain
importlib.reload(pretrain)
pretrain.main(args_list=[
    'configs/pretrain.yaml',
    '--loss_ablation_mode', 'gen_all',
    '--vision_unfreeze_last_n', '2',
    '--mlm_mask_mode', 'wholeword',
    '--metric_for_best_model', 'acc_mlm_grounded',
    '--num_train_epochs', '5',
])
# Mỗi 1050 step lưu checkpoint -> Cell 3 (upload) rồi Cell 2b (resume) khi hết phiên.

### 2b) TRUE-RESUME — tiếp tục ĐÚNG như train liền mạch tới 10 epoch
Điều kiện: run gốc đã đặt **`num_train_epochs=10` + constant LR**.
TRUE-resume khôi phục **optimizer + scheduler + RNG + data-skip** → chạy tiếp y như chưa dừng.
- **Giữ `num_train_epochs=10`** (KHÔNG đổi).
- `REPO` = repo checkpoint của **run constant-10 này** (đừng lẫn checkpoint cosine-3 cũ).
- Lặp lại cell này sau mỗi lần Colab ngắt cho tới khi đủ 10 epoch.

In [ ]:
# === TRUE-RESUME run V4 ===
# env + args PHẢI KHỚP Y HỆT Cell 9 (VISION_LR_SCALE quyết định cấu trúc param-group
# optimizer; lệch là load optimizer.pt LỖI). ITC queue không nằm trong checkpoint ->
# sau resume tự đầy lại sau ~K/B step (xấp xỉ chấp nhận được).
import os, importlib
from huggingface_hub import list_repo_files, snapshot_download
REPO = 'Kus669/ViSceT5-pretrain-genall-v4'
OUT  = '/content/ViSceT5/output/pretrain'
files = list_repo_files(REPO)
ckpts = sorted({f.split('/')[0] for f in files if f.startswith('checkpoint-')}, key=lambda x:int(x.split('-')[1]))
assert ckpts, 'Repo chưa có checkpoint-* (chạy Cell 9 + Cell 3 upload trước).'
latest = ckpts[-1]
snapshot_download(REPO, repo_type='model', allow_patterns=[f'{latest}/*'], local_dir=OUT)
resume_path = os.path.join(OUT, latest)
print('True-resume từ:', resume_path)
# --- KHỚP Y HỆT Cell 9 ---
os.environ['ITC_WEIGHT']='1'
os.environ['ITC_TEXT_SOURCE']='ocr'  # v4: image<->chuỗi-OCR (câu hỏi template nghèo tin -> ITC v3 ghim ln4;
                                     #     OCR định danh ảnh duy nhất + ép CLIP học ĐỌC chữ)
os.environ['ITC_QUEUE']='1024'        # v4: hạ 4096->1024 — đủ khó, thấy tiến độ học sớm hơn
os.environ['ITC_DUP_TAU']='1'         # mask false-neg: trùng NỘI-DUNG-text (hash) HOẶC CÙNG-ẢNH (image_id)
os.environ['TWC_ADV_PROB']='0.6'
os.environ['TWC_DUP_BOX']='0'
os.environ['ITM_WEIGHT']='0'
os.environ['ITM_POLLUTE']='0'
os.environ['GEN_TARGET_STYLE']='qa'
os.environ['VISION_LR_SCALE']='0.1'   # đổi cấu trúc param-group optimizer -> resume PHẢI khớp
os.environ['MLM_RAND_PROB']='0.25'
os.environ.pop('ITC_TEXT_POOL', None) # không dùng ở mode ocr
os.environ.pop('TWC_TRAIN_LOG', None)
from training import pretrain; importlib.reload(pretrain)
pretrain.main(args_list=[
    'configs/pretrain.yaml',
    '--loss_ablation_mode', 'gen_all',
    '--vision_unfreeze_last_n', '2',
    '--mlm_mask_mode', 'wholeword',
    '--metric_for_best_model', 'acc_mlm_grounded',
    '--num_train_epochs', '5',         # GIỮ NGUYÊN như Cell 9
    '--resume_from_checkpoint', resume_path,
])
# XONG -> Cell 3 (upload) để lần sau resume tiếp.

### 3) Upload model pretrain lên HF (để finetune_colab.ipynb dùng)

In [ ]:
import os, re
from huggingface_hub import HfApi
OUT = '/content/ViSceT5/output/pretrain'
cks = [d for d in os.listdir(OUT) if re.match(r'checkpoint-\d+$', d)]
assert cks, 'Chưa có checkpoint nào trong output/pretrain.'
latest = max(cks, key=lambda d: int(d.split('-')[1]))
api = HfApi(token=os.environ['HF_TOKEN'])
api.create_repo(repo_id=HF_PRETRAIN_REPO, repo_type='model', exist_ok=True)
# Upload TRỌN checkpoint mới nhất, BẮT BUỘC kèm optimizer.pt — thiếu nó thì lần
# resume sau HF Trainer âm thầm tạo optimizer MỚI (mất momentum) = không còn true-resume.
api.upload_folder(folder_path=os.path.join(OUT, latest), path_in_repo=latest,
                  repo_id=HF_PRETRAIN_REPO, repo_type='model')
print('Uploaded', latest, '->', HF_PRETRAIN_REPO)